# Gold — Product Dimension
One row per **current** product (`end_date IS NULL`). Historical versions are dropped for now; category info comes from the ERP lookup.

→ `gold.dim_products`

## Transformation (SQL)

In [ ]:
CATALOG = "workspace"

query = f"""
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_key,   -- surrogate key
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_cost,
    pn.product_line,
    pn.start_date
FROM {CATALOG}.silver.crm_products pn
LEFT JOIN {CATALOG}.silver.erp_product_category pc
    ON pn.category_id = pc.category_id
WHERE pn.end_date IS NULL
"""

df = spark.sql(query)

## Sanity check

In [ ]:
df.limit(10).display()

## Write gold table

In [ ]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_products")

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM workspace.gold.dim_products;